<a href="https://colab.research.google.com/github/norahmunoz/ds2002-fa26/blob/main/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [34]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone

BASE = "https://archive-api.open-meteo.com/v1/archive"
RETRY_STATUSES = {429}


def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Get daily maximum temperature and total precipitation."""

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start,
        "end_date": end,
        "daily": "temperature_2m_max,precipitation_sum",
        "timezone": "America/New_York",
    }

    if tries < 1:
        raise ValueError("tries must be at least 1")

    last_error = None

    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)

            if r.status_code == 200:
                return pd.DataFrame(r.json()["daily"])

            # Raise immediately for errors that should not be retried.
            if (
                r.status_code not in RETRY_STATUSES
                and not (500 <= r.status_code <= 599)
            ):
                raise RuntimeError(
                    f"Not retryable: {r.status_code} {r.text[:200]}"
                )

            # Save retryable HTTP errors.
            last_error = RuntimeError(
                f"HTTP {r.status_code}: {r.text[:200]}"
            )

        except requests.RequestException as e:
            last_error = e

        # Retry with an increasing delay.
        if attempt < tries - 1:
            delay = 0.5 * (2 ** attempt)
            print(
                f"Attempt {attempt + 1} failed ({last_error}); "
                f"retrying in {delay} seconds"
            )
            time.sleep(delay)

    raise RuntimeError(
        f"Weather request failed after {tries} attempts: {last_error}"
    ) from last_error

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [35]:
used_fallback = False

try:
    weather_all = get_weather("2026-09-05", "2026-10-03")

    wx = weather_all[
        weather_all["time"].isin(["2026-09-05", "2026-10-03"])
    ].copy()

    if len(wx) != 2:
        raise ValueError("API did not return both requested dates")

    wx["source"] = "open-meteo"

except Exception as e:
    print("Weather request failed:", e)
    wx = None

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [36]:
FALLBACK = pd.DataFrame({
    "time": ["2026-09-05", "2026-10-03"],
    "temperature_2m_max": [30.1, 18.4],
    "precipitation_sum": [0.0, 22.5],
})

if used_fallback or wx is None:
    used_fallback = True
    wx = FALLBACK.copy()
    wx["source"] = "fallback sample"

display(wx)

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [37]:
wx = wx.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "precipitation_sum": "precip_mm",
})

wx["date"] = pd.to_datetime(wx["date"])

wx = wx[["date", "temp_max", "precip_mm", "source"]]

wx = wx.sort_values("date").reset_index(drop=True)

display(wx)

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [38]:
assert list(wx.columns) == [
    "date", "temp_max", "precip_mm", "source"
]

assert len(wx) == 2, f"expected 2 rows, got {len(wx)}"

assert pd.api.types.is_datetime64_any_dtype(wx["date"])

assert wx["precip_mm"].notna().all(), "missing precipitation"

assert (wx["precip_mm"] >= 0).all(), "negative rainfall is not a thing"

assert wx["date"].dt.strftime("%Y-%m-%d").tolist() == [
    "2026-09-05", "2026-10-03"
]

assert wx["source"].isin(
    ["open-meteo", "fallback sample"]
).all()

print("weather table is safe to join.")

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [39]:
sep5 = wx.loc[
    wx["date"] == pd.Timestamp("2026-09-05"), "precip_mm"
].iloc[0]

oct3 = wx.loc[
    wx["date"] == pd.Timestamp("2026-10-03"), "precip_mm"
].iloc[0]

difference = abs(oct3 - sep5)

if difference == 0:
    print(
        "September 5 and October 3 had the same total precipitation."
    )
elif oct3 > sep5:
    print(
        f"October 3 was wetter, with {difference:.1f} mm more "
        "precipitation than September 5."
    )
else:
    print(
        f"September 5 was wetter, with {difference:.1f} mm more "
        "precipitation than October 3."
    )

October 3 was wetter, with 12.2 mm more precipitation than September 5.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [40]:
pulled_at = datetime.now(timezone.utc).isoformat()

print("Source: Open-Meteo historical archive API")
print("Endpoint:", BASE)
print("Coordinates: latitude 38.03, longitude -78.51")
print("Date range: 2026-09-05 to 2026-10-03")
print("Pulled at (UTC):", pulled_at)
print("Used fallback:", used_fallback)
print("Data source:", wx["source"].iloc[0])
print(
    "Documentation: "
    "https://open-meteo.com/en/docs/historical-weather-api"
)

Source: Open-Meteo historical archive API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: latitude 38.03, longitude -78.51
Date range: 2026-09-05 to 2026-10-03
Pulled at (UTC): 2026-10-10T19:14:37.669058+00:00
Used fallback: False
Data source: open-meteo
Documentation: https://open-meteo.com/en/docs/historical-weather-api


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame, the code would still run, but the weather values would be missing for all 200 days of sales. The chart might make it look like rain has no effect on revenue. Nobody might notice because there would be no error message, so they could mistake missing data for a real result.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [41]:
cities = {
    "Charlottesville": (38.03, -78.51),
    "Orlando": (28.54, -81.38),
}

city_frames = []

for city, (lat, lon) in cities.items():
    city_weather = get_weather(
        "2026-09-05",
        "2026-10-03",
        lat=lat,
        lon=lon,
    )

    city_weather = city_weather[
        city_weather["time"].isin(["2026-09-05", "2026-10-03"])
    ].copy()

    if len(city_weather) != 2:
        raise ValueError(f"Missing requested dates for {city}")

    city_weather["city"] = city
    city_frames.append(city_weather)

stretch = pd.concat(city_frames, ignore_index=True)

stretch = stretch.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "precipitation_sum": "precip_mm",
})

stretch["date"] = pd.to_datetime(stretch["date"])

stretch = stretch[
    ["date", "city", "temp_max", "precip_mm"]
]

stretch = stretch.sort_values(
    ["date", "city"]
).reset_index(drop=True)

assert len(stretch) == 4
assert stretch["precip_mm"].notna().all()
assert (stretch["precip_mm"] >= 0).all()

display(stretch)

,date,city,temp_max,precip_mm
0,2026-09-05,Charlottesville,29.4,2.5
1,2026-09-05,Orlando,32.1,3.9
2,2026-10-03,Charlottesville,22.0,14.7
3,2026-10-03,Orlando,30.4,14.5
